# **Main — Analysis Pipeline**

## **1. Purpose and Execution Order**

This orchestrator runs the study through a single **Run All** action. Each notebook executes in its own Python kernel, using the same environment as main.

Default order: **00 → 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08**. Notebook 09 presents the results synthesis and does not require execution.

Source notebooks remain unchanged. Executed copies are saved under `results/pipeline_runs/`, while each analysis writes its usual outputs to its configured results folder. Existing caches are reused according to each notebook's settings; missing results can trigger long model fits.

A failure stops the pipeline and saves the partial notebook and error status. Review notebook 00 for the research question, dataset dictionary, preprocessing, scaling, dependencies, and reproducibility decisions.

## **2. Configuration**

Select the project Python environment as the kernel. Set `RUN_PREPROCESSING = False` only if the processed dataset is already available and current. The list below controls the analysis order.

In [ ]:
from pathlib import Path
from datetime import datetime
from time import perf_counter
import sys

import nbformat
from nbclient import NotebookClient

working_directory = Path.cwd().resolve()
candidates = [working_directory, *working_directory.parents]
candidates += [p / "Corte_3" / "PROYECTO_FINAL" for p in candidates.copy()]
PROJECT_ROOT = next(
    (p for p in candidates
     if (p / "src" / "config.py").is_file()
     and (p / "notebooks" / "07_neural_networks_autoencoders_vae.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open main.ipynb from the project or course workspace.")

RUN_PREPROCESSING = True
CELL_TIMEOUT_SECONDS = None  # Allow long training cells to finish.
NOTEBOOKS = [
    "01_problem_data_reproducibility.ipynb",
    "02_gmm_latent_variables_em.ipynb",
    "03_svd_pca_probabilistic_pca_lsa.ipynb",
    "04_recommendation_ica_matrix_factorization.ipynb",
    "05_manifolds_tsne_umap_embeddings.ipynb",
    "06_anomaly_detection.ipynb",
    "07_neural_networks_autoencoders_vae.ipynb",
    "08_support_confidence_lift_and_rules.ipynb",
]
if RUN_PREPROCESSING:
    NOTEBOOKS.insert(0, "00_cleaning_and_preprocessing.ipynb")

print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Execution order:")
for name in NOTEBOOKS:
    print(" •", name)

## **3. Notebook Execution and Progress**

The execution helper counts completed code cells, stops at errors, and saves the output even when execution fails. Each child uses a separate kernel, so variables do not carry over between notebooks.

In [ ]:
from jupyter_client.kernelspec import KernelSpecManager
import json


def run_notebook(source, destination, kernel_name):
    notebook = nbformat.read(source, as_version=4)
    # Remove stale outputs so failed runs cannot show old results as new ones.
    for cell in notebook.cells:
        if cell.cell_type == "code":
            cell.outputs = []
            cell.execution_count = None

    total = sum(cell.cell_type == "code" and bool(cell.source.strip())
                for cell in notebook.cells)
    progress = {"completed": 0}

    def on_cell_executed(cell, cell_index, **kwargs):
        if cell.cell_type == "code" and cell.source.strip():
            progress["completed"] += 1
            print(f"  Cells completed: {progress['completed']}/{total}", flush=True)

    client = NotebookClient(
        notebook,
        kernel_name=kernel_name,
        kernel_manager_class=PipelineKernelManager,
        timeout=CELL_TIMEOUT_SECONDS,
        allow_errors=False,
        resources={"metadata": {"path": str(PROJECT_ROOT)}},
        on_cell_executed=on_cell_executed,
    )
    try:
        client.execute()
    finally:
        # Save completed outputs, including the failing cell if execution failed.
        nbformat.write(notebook, destination)


## **4. Run the Pipeline**

Run this cell after the configuration and helper cells, or select **Run All**. Each notebook prints `SUCCESS` when complete. Inspect its executed copy for detailed tables, figures, and cell outputs. Preprocessing rewrites the processed CSV; cached analyses may be loaded instead of retrained.

In [ ]:
import tempfile

# A temporary kernel specification ensures children use the same Python as main.
# It does not install or alter the user's registered kernels.
missing = [name for name in NOTEBOOKS
           if not (PROJECT_ROOT / "notebooks" / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing notebooks: {missing}")

RUN_DIR = PROJECT_ROOT / "results" / "pipeline_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
summary = []
pipeline_start = perf_counter()
print("Executed notebooks and status:", RUN_DIR, flush=True)

with tempfile.TemporaryDirectory(prefix="analysis_pipeline_kernel_") as kernel_root:
    kernel_name = "pipeline-python"
    kernel_dir = Path(kernel_root) / kernel_name
    kernel_dir.mkdir()
    (kernel_dir / "kernel.json").write_text(json.dumps({
        "argv": [sys.executable, "-m", "ipykernel_launcher", "-f", "{connection_file}"],
        "display_name": "Pipeline Python",
        "language": "python",
    }))
    # Supply this manager to nbclient's kernel manager without changing the environment.
    from jupyter_client import AsyncKernelManager

    class PipelineKernelManager(AsyncKernelManager):
        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self.kernel_spec_manager = KernelSpecManager(kernel_dirs=[kernel_root])

    # Each client receives the manager explicitly; no global or installed kernel changes.
    for position, name in enumerate(NOTEBOOKS, start=1):
        start = perf_counter()
        destination = RUN_DIR / name
        print(f"\n[{position}/{len(NOTEBOOKS)}] Running {name}", flush=True)
        status = {"notebook": name, "output": str(destination)}
        try:
            run_notebook(PROJECT_ROOT / "notebooks" / name, destination, kernel_name)
        except BaseException as error:
            status.update(status="failed", seconds=round(perf_counter() - start, 2),
                          error=f"{type(error).__name__}: {error}")
            summary.append(status)
            (RUN_DIR / "status.json").write_text(json.dumps(summary, indent=2))
            print(f"FAILED: {name}. Partial results: {destination}", flush=True)
            raise
        else:
            status.update(status="success", seconds=round(perf_counter() - start, 2))
            summary.append(status)
            (RUN_DIR / "status.json").write_text(json.dumps(summary, indent=2))
            print(f"SUCCESS: {name} ran successfully ({status['seconds']:.1f} s).", flush=True)

print(f"\nAll {len(NOTEBOOKS)} notebooks ran successfully in {(perf_counter() - pipeline_start) / 60:.1f} minutes.")